In [ ]:
# Sf1 decoding deviation

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib as mpl
from paths import DATA_DIR, fig_dir


In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
from spyglass.common import Session

In [ ]:
# custom schema
from find_my_data import *
from alison_decoding import ClusterlessAcausalResultsSummary

In [ ]:
from fig_helpers import *
set_figure_defaults()

In [ ]:
save_fig = False
fig_path = fig_dir('figs26')
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))


### load data

In [ ]:
# Params
behavior_model_params_name = 'default_hmm_0623' #'hmm_test' #'default_hmm'

position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

In [ ]:
out_path = f'{DATA_DIR}/big_df_pkls/'
# today_now = datetime.now().strftime("%Y%m%d") 
today_now = '20240212'
subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut']

In [ ]:
big_dfs = {}
for subject_id in subject_ids:
    try:
#         if subject_id == 'senor':
#             senor_big_df = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'chimi':
#             chimi_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'wilbur':
#             wilbur_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'peanut':
#             peanut_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'j16':
#             j16_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_uncertainty_'+today_now+'.pkl')
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

In [ ]:
stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')

    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]

print(stable_clusterless_nwbs)

In [ ]:
is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}

# get to stable data only
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable

In [ ]:
for subject_id in subject_ids:
    df = all_rat_big_dfs_stable[subject_id]
    p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]
    all_rat_big_dfs_stable[subject_id] = all_rat_big_dfs_stable[subject_id][~all_rat_big_dfs_stable[subject_id][p_rew_cols].eq(all_rat_big_dfs_stable[subject_id]['p_rew_leaf1'], axis=0).all(axis=1)]

In [ ]:
all_rat_big_dfs_stable['j16']

In [ ]:
all_rat_big_dfs_stable['j16'].columns[0:60],all_rat_big_dfs_stable['j16'].columns[60:120],all_rat_big_dfs_stable['j16'].columns[120:180],all_rat_big_dfs_stable['j16'].columns[180:]

In [ ]:
# Make all animal concatenated df
concatenated_df0 = pd.concat(all_rat_big_dfs_stable.values(), keys=all_rat_big_dfs_stable.keys(), names=['subject_id'])
concatenated_df0

In [ ]:
# Reset the index to make subject_id a column
concatenated_df0.reset_index(level=0, inplace=True)
concatenated_df0

In [ ]:
concatenated_df_with_time = concatenated_df0.reset_index(drop=False, inplace=False)
concatenated_df_with_time

In [ ]:
concatenated_df = concatenated_df_with_time

### prep plot fxn

In [ ]:
fig_path = fig_dir('decodingerror26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)

In [ ]:
from fig_helpers import *
set_figure_defaults()
    

In [ ]:
def actual_vs_decoded_2d_hist_normalized_per_rat(concatenated_df, subject_ids,                                                  
                                                  figwidth, figheight, fig_path,
                                                 cmap = 'magma',
                                                  bad = 'black',
                                                  n_bins=200,
                                                  vmax = 1,
                                                  save_fig = False,
                                                 ):

    for subject_id in subject_ids:
        big_df_firstgraph = all_rat_big_dfs_stable[subject_id]

        fig1, ax1 = plt.subplots(ncols=1,figsize=(6,5))
        hist1, xedges1, yedges1, mesh1 = ax1.hist2d(big_df_firstgraph['linear_position'],big_df_firstgraph['max_posterior_acausal'],bins=n_bins) #, vmax=100)
        ax1.set_xlabel('actual linpos')
        ax1.set_ylabel('peak posterior')
        ax1.set_title(f'counts_firstgraph\n')
        current_cmap = plt.cm.get_cmap()
        current_cmap.set_bad(color=bad)
        fig1.colorbar(mesh1,label='counts per linpos bin')
        plt.show()

        fig2, ax2 = plt.subplots(ncols=1,figsize=(figwidth,figheight))
        #plt.figure(figsize=(10,10))
        #fig2, ax2 = plt.subplots(ncols=1)

        column_totals = hist1.sum(axis=1, keepdims=True)
        #print(column_totals)
        hist_prop_of_actual_time = hist1/column_totals
        mesh2 = ax2.pcolormesh(xedges1,yedges1, hist_prop_of_actual_time.T, vmax=vmax)

        ax2.set_xlabel('Actual Position  [cm]')
        ax2.set_ylabel('Decoded Position [cm]')
        ax2.set_title(f'Rat {subject_id[0].upper()}, Normalized, vmax={vmax}, bins={n_bins}')
        fig2.colorbar(mesh2,label='Proportion of Actual Position Time', shrink=.8)
        ax2.set_aspect('equal', adjustable='box')
        if save_fig:
            fig_name = f'{subject_id}_actual_vs_decoded_hist_vmax{vmax}_cmap{cmap}_bad{bad}_nbins{n_bins}_w{figwidth}_h{figheight}'
            save_figure(fig_path, fig_name)

        fig2.show()
    


### plot decoding deviation

In [ ]:
cmap = 'magma'
bad = 'black'
vmax=1
n_bins = 200
figwidth = TWO_COLUMN
figheight = figwidth
for vmax in [0.2]:#[1,.5,.2,.1,.05]:
    actual_vs_decoded_2d_hist_normalized_per_rat(concatenated_df, subject_ids,                                                  
                                                  figwidth, figheight, fig_path,
                                                 cmap = cmap,
                                                  bad = bad,
                                                  n_bins=n_bins,
                                                  vmax = vmax,
                                                  save_fig = save_fig,
                                                 )

In [ ]:
# Make all animal concatenated df
concatenated_df = pd.concat(all_rat_big_dfs_stable.values(), keys=all_rat_big_dfs_stable.keys(), names=['subject_id'])

# Reset the index to make subject_id a column
concatenated_df.reset_index(level=0, inplace=True)
concatenated_df.reset_index(drop=True, inplace=True)
concatenated_df

In [ ]:
# save_fig = False

# custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))


# median_lw=3
# fliersize=0
# figwidth = 8
# figheight = figwidth*GOLDEN_RATIO

# plot_decode_to_actual_pos_distance_by_rat(concatenated_df, subject_ids,figwidth=figwidth, figheight=figheight, fig_path=fig_path,
#                                             rat_colors=custom_colors_by_rat, median_lw=median_lw, fliersize=fliersize,
#                                             save_fig = save_fig,
#                                                  )

In [ ]:
from fig_helpers import *
set_figure_defaults()
#ylim was 53 before
def plot_decode_to_actual_pos_distance_by_rat(concatenated_df, subject_ids, figwidth, figheight, fig_path,
                                              rat_colors=iter(cm.tab20b([0,.8, .85, .1, .05])),median_lw=3, fliersize=0,
                                                  
                                                  save_fig = False,
                                                 ):
    plt.figure(figsize=(figwidth,figheight))  # Adjust the figure size as needed
    sns.boxplot(x='subject_id', y='abs_ahead_behind_distance', data=concatenated_df, fliersize=fliersize,
                palette=rat_colors,
               medianprops={'linewidth':median_lw})
    medians = concatenated_df.groupby('subject_id')['abs_ahead_behind_distance'].median().values
    for i, median in enumerate(medians):
        plt.text(i, 25, f'{median:.2f}', horizontalalignment='center', size='medium', color='black')
    plt.xlabel('Subject', y=-.15)
    plt.xticks([0,1,2,3,4],[f'Rat {subject_id[0].upper()}' for subject_id in subject_ids])
    plt.ylabel('Decoded-to-Actual Position Distance [cm]')
    plt.ylim(0,30)
    sns.despine()
    fig_name = f'all_rat_decode_to_actual_pos_distance_fliersize{fliersize}_medianlw{median_lw}_w{figwidth}_h{figheight}'
    if save_fig:
        save_figure(fig_path,fig_name)
    plt.show()

# save_fig = False

custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

fig_path = fig_dir('DraftFigs/june_decoding_exs/decoding_error')

median_lw=3
fliersize=0
figwidth = 8
figheight = figwidth*GOLDEN_RATIO

plot_decode_to_actual_pos_distance_by_rat(concatenated_df, subject_ids,figwidth=figwidth, figheight=figheight, fig_path=fig_path,
                                            rat_colors=custom_colors_by_rat, median_lw=median_lw, fliersize=fliersize,
                                            save_fig = save_fig,
                                                 )
